In [ ]:
# import, fix font

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

import scienceplots
plt.style.use(['science', 'ieee', 'bright'])
plt.rcParams.update({"text.usetex": True, "font.family": "serif", "font.serif": ["Computer Modern"]})
plt.rcParams["axes.unicode_minus"] = False

Population Plots

In [ ]:
# z_hor against chirp mass (fig 1 in new paper)

import numpy as np
import json
from pathlib import Path

dir = Path("/Users/natalieyyyyy/Projects")

z_hor_grid_mean = np.load(dir / "z_hor_grid_mean.npy", allow_pickle=True)
chirp_mass_grid = np.load(dir / "chirp_mass_grid.npy", allow_pickle=True)

posterior_samples_list = []
n_events = 100  # Change this to 1486 later

# Load existing dictionary
with open(dir / "unlensed_param_detectable.json", "r") as f:
    unlensed_param_detectable = json.load(f)

# Extract component masses
m1 = np.asarray(unlensed_param_detectable["mass_1"])
m2 = np.asarray(unlensed_param_detectable["mass_2"])

# Calculate chirp mass
Mc = (m1 * m2)**(3/5) / (m1 + m2)**(1/5)

# Add chirp mass to existing dictionary
unlensed_param_detectable["chirp_mass"] = Mc.tolist()

plt.figure(figsize=(8, 6))

plt.plot(chirp_mass_grid, z_hor_grid_mean)
plt.scatter(unlensed_param_detectable["chirp_mass"], unlensed_param_detectable["zs"], s=10, alpha=0.5, label='Detected events')

plt.xlabel(r"Chirp mass $\mathcal{M}\ [M_\odot]$")
plt.ylabel(r"Horizon redshift $z_\text{hor}$")

plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(dir / "results/z_hor_vs_chirp_mass.pdf", dpi=300, bbox_inches="tight")


In [ ]:
# weight against chirp mass (fig 2 in new paper; less important?)

import numpy as np

w_grid = np.load("/Users/natalieyyyyy/Projects/w_grid.npy", allow_pickle=True)
chirp_mass_grid = np.load("/Users/natalieyyyyy/Projects/chirp_mass_grid.npy", allow_pickle=True)

plt.figure(figsize=(8, 6))

plt.plot(chirp_mass_grid, np.log10(w_grid))

plt.xlabel(r"Chirp mass $\mathcal{M}\ [M_\odot]$")
plt.ylabel(r"$\log_{10}(w)$")

plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig("/Users/natalieyyyyy/Projects/results/log_weight_vs_chirp_mass.pdf", dpi=300, bbox_inches="tight")

There are many versions of code for the detectable and intrinsic population plots, as the way we present them are not finalised yet, so I am keeping all versions in case I need them.

In [ ]:
# not needed unless log of mass too ugly

# population density (3d)
# substitute population density (2d) as redshift should be included
# v1: component masses

import matplotlib.texmanager as mtex
# mtex.TexManager.is_usable = lambda args, *kwargs: False
import matplotlib as mpl
# mpl.rcParams['text.usetex'] = False
import numpy as np
from figaro.plot import plot_multidim

from matplotlib.ticker import MaxNLocator, AutoMinorLocator

print("Loading event samples and population densities", flush=True)

hier_draws = np.load("/Users/natalieyyyyy/Projects/intermediates/population_densities_3d.npy", allow_pickle = True)
posterior_samples_list = np.load("/Users/natalieyyyyy/Projects/intermediates/posterior_samples_list.npy", allow_pickle = True)

print("Loading figure...", flush=True)

fig = plot_multidim(hier_draws, samples = np.concatenate(posterior_samples_list), 
                    labels = [r"$\log_{10}(m_1)$", r"$\log_{10}(m_2)$", r"\mathrm{redshift}"], show = False, median_label=r'\mathrm{HDPGMM}')

axs = np.array(fig.axes).reshape(3, 3)

# fig.suptitle(r"$\mathrm{Component\ Masses\ Population}$", fontsize=18, y=1.02)
fig.subplots_adjust(top=0.98, wspace=0.07, hspace=0.07)

# Set font sizes for axes labels and ticks
for ax in fig.axes:
    ax.xaxis.label.set_size(14)
    ax.yaxis.label.set_size(14)
    ax.tick_params(axis='both', labelsize=12)

# Adjust the limits of the diagonal and off-diagonal plots based on the data
for i in range(3):
    all_samples = np.concatenate([s[:, i] for s in posterior_samples_list])

    xmin = np.percentile(all_samples, 1)
    xmax = np.percentile(all_samples, 99)
    axs[i, i].set_xlim(xmin, xmax)
    ymax = max(line.get_ydata().max() for line in axs[i, i].lines)
    axs[i, i].set_ylim(0, 1.05 * ymax)

    # Set number of major ticks on the parameter axis
    axs[i, i].xaxis.set_major_locator(MaxNLocator(nbins=5))
    axs[i, i].xaxis.set_minor_locator(AutoMinorLocator(2))

    for j in range(i + 1, 3):
        axs[j, i].set_xlim(xmin, xmax)

        # x-axis is the same parameter, so use the same ticks
        axs[j, i].xaxis.set_major_locator(MaxNLocator(nbins=5))
        axs[j, i].xaxis.set_minor_locator(AutoMinorLocator(2))

    for j in range(i + 1, 3):
        axs[j, i].set_xlim(xmin, xmax)
        # x-axis is the same parameter, so use the same ticks
        axs[j, i].xaxis.set_major_locator(MaxNLocator(nbins=5))
        axs[j, i].xaxis.set_minor_locator(AutoMinorLocator(2))
    for j in range(i):
        axs[i, j].set_ylim(xmin, xmax)
        # y-axis is the same parameter, so use the same ticks
        axs[i, j].yaxis.set_major_locator(MaxNLocator(nbins=5))
        axs[i, j].yaxis.set_minor_locator(AutoMinorLocator(2))

# Set font sizes for legend texts
for ax in fig.axes:
    if ax.legend_ is not None:
        for text in ax.legend_.get_texts():
            text.set_fontsize(14)

fig.savefig("/Users/natalieyyyyy/Projects/unlensed_3d.pdf", dpi=300, bbox_inches="tight")
fig.show()

print("Plot saved to Projects")



In [ ]:
# population density (3d) (fig 4 left in new paper)
# substitute population density (2d) as redshift should be included
# v2: log of component masses

import matplotlib.texmanager as mtex
# mtex.TexManager.is_usable = lambda args, *kwargs: False
import matplotlib as mpl
# mpl.rcParams['text.usetex'] = False
import numpy as np
from figaro.plot import plot_multidim

from matplotlib.ticker import MaxNLocator, AutoMinorLocator

print("Loading event samples and population densities", flush=True)

hier_draws = np.load("/Users/natalieyyyyy/Projects/intermediates/population_densities_3d_log.npy", allow_pickle = True)
posterior_samples_list = np.load("/Users/natalieyyyyy/Projects/intermediates/posterior_samples_list_log.npy", allow_pickle = True)

print("Loading figure...", flush=True)

fig = plot_multidim(hier_draws, samples = np.concatenate(posterior_samples_list), 
                    labels = [r'm_1', r'm_2', r'z_s'], show = False, median_label=r'\mathrm{HDPGMM}')

axs = np.array(fig.axes).reshape(3, 3)

# fig.suptitle(r"$\mathrm{Component\ Masses\ Population}$", fontsize=18, y=1.02)
fig.subplots_adjust(top=0.98, wspace=0.07, hspace=0.07)

# Set font sizes for axes labels and ticks
for ax in fig.axes:
    ax.xaxis.label.set_size(14)
    ax.yaxis.label.set_size(14)
    ax.tick_params(axis='both', labelsize=12)

# Adjust the limits of the diagonal and off-diagonal plots based on the data
for i in range(3):
    all_samples = np.concatenate([s[:, i] for s in posterior_samples_list])

    xmin = np.percentile(all_samples, 1)
    xmax = np.percentile(all_samples, 99)
    axs[i, i].set_xlim(xmin, xmax)
    ymax = max(line.get_ydata().max() for line in axs[i, i].lines)
    axs[i, i].set_ylim(0, 1.05 * ymax)

    # Set number of major ticks on the parameter axis
    axs[i, i].xaxis.set_major_locator(MaxNLocator(nbins=5))
    axs[i, i].xaxis.set_minor_locator(AutoMinorLocator(2))

    for j in range(i + 1, 3):
        axs[j, i].set_xlim(xmin, xmax)

        # x-axis is the same parameter, so use the same ticks
        axs[j, i].xaxis.set_major_locator(MaxNLocator(nbins=5))
        axs[j, i].xaxis.set_minor_locator(AutoMinorLocator(2))

    for j in range(i + 1, 3):
        axs[j, i].set_xlim(xmin, xmax)
        # x-axis is the same parameter, so use the same ticks
        axs[j, i].xaxis.set_major_locator(MaxNLocator(nbins=5))
        axs[j, i].xaxis.set_minor_locator(AutoMinorLocator(2))
    for j in range(i):
        axs[i, j].set_ylim(xmin, xmax)
        # y-axis is the same parameter, so use the same ticks
        axs[i, j].yaxis.set_major_locator(MaxNLocator(nbins=5))
        axs[i, j].yaxis.set_minor_locator(AutoMinorLocator(2))

# Set font sizes for legend texts
for ax in fig.axes:
    if ax.legend_ is not None:
        for text in ax.legend_.get_texts():
            text.set_fontsize(14)

fig.savefig("/Users/natalieyyyyy/Projects/results/unlensed_3d_log.pdf", dpi=300, bbox_inches="tight")

print("Plot saved")



In [ ]:
# not needed as switched to 3d

# population density (2d)

import matplotlib.texmanager as mtex
# mtex.TexManager.is_usable = lambda args, *kwargs: False
import matplotlib as mpl
# mpl.rcParams['text.usetex'] = False
import numpy as np
from figaro.plot import plot_multidim

from matplotlib.ticker import MaxNLocator, AutoMinorLocator

print("Loading event samples and population densities", flush=True)

hier_draws = np.load("/Users/natalieyyyyy/Projects/intermediates/population_densities_2d.npy", allow_pickle = True)
posterior_samples_list = np.load("/Users/natalieyyyyy/Projects/intermediates/posterior_samples_list.npy", allow_pickle = True)

print("Loading figure...", flush=True)

fig = plot_multidim(hier_draws, samples = np.concatenate(posterior_samples_list), 
                    labels = [r'm_1', r'm_2'], show = False, median_label=r'\mathrm{HDPGMM}')

axs = np.array(fig.axes).reshape(2, 2)

# fig.suptitle(r"$\mathrm{Component\ Masses\ Population}$", fontsize=18, y=1.02)
fig.subplots_adjust(top=0.98, wspace=0.07, hspace=0.07)

# Set font sizes for axes labels and ticks
for ax in fig.axes:
    ax.xaxis.label.set_size(14)
    ax.yaxis.label.set_size(14)
    ax.tick_params(axis='both', labelsize=12)

# Adjust the limits of the diagonal and off-diagonal plots based on the data
for i in range(2):
    all_samples = np.concatenate([s[:, i] for s in posterior_samples_list])

    xmin = np.percentile(all_samples, 1)
    xmax = np.percentile(all_samples, 99)
    axs[i, i].set_xlim(xmin, xmax)
    ymax = max(line.get_ydata().max() for line in axs[i, i].lines)
    axs[i, i].set_ylim(0, 1.05 * ymax)

    # Set number of major ticks on the parameter axis
    axs[i, i].xaxis.set_major_locator(MaxNLocator(nbins=5))
    axs[i, i].xaxis.set_minor_locator(AutoMinorLocator(2))

    for j in range(i + 1, 2):
        axs[j, i].set_xlim(xmin, xmax)

        # x-axis is the same parameter, so use the same ticks
        axs[j, i].xaxis.set_major_locator(MaxNLocator(nbins=5))
        axs[j, i].xaxis.set_minor_locator(AutoMinorLocator(2))

    for j in range(i + 1, 2):
        axs[j, i].set_xlim(xmin, xmax)
        # x-axis is the same parameter, so use the same ticks
        axs[j, i].xaxis.set_major_locator(MaxNLocator(nbins=5))
        axs[j, i].xaxis.set_minor_locator(AutoMinorLocator(2))
    for j in range(i):
        axs[i, j].set_ylim(xmin, xmax)
        # y-axis is the same parameter, so use the same ticks
        axs[i, j].yaxis.set_major_locator(MaxNLocator(nbins=5))
        axs[i, j].yaxis.set_minor_locator(AutoMinorLocator(2))

# Set font sizes for legend texts
for ax in fig.axes:
    if ax.legend_ is not None:
        for text in ax.legend_.get_texts():
            text.set_fontsize(14)

fig.savefig("/Users/natalieyyyyy/Projects/unlensed_2d.pdf", dpi=300, bbox_inches="tight")
fig.show()

print("Plot saved to Projects")



In [ ]:
# population density (3d; reweighted) (fig 4 right in new paper)
# substitute population density (2d) as redshift should be included
# v2: log of component masses

import numpy as np
from scipy.interpolate import interp1d
from pathlib import Path
from figaro.mixture import DPGMM
from figaro.utils import get_priors
from figaro.plot import plot_multidim
from tqdm import tqdm

from figaro.utils import rvs_median
from ler import LeR
from ler.gw_source_population import CBCSourceRedshiftDistribution
ler = LeR(spin_precession=True)

from matplotlib.ticker import MaxNLocator, AutoMinorLocator

np.random.seed(42)

population_densities_3d = np.load("/Users/natalieyyyyy/Projects/intermediates/population_densities_3d_log.npy", allow_pickle=True)
cbc = CBCSourceRedshiftDistribution(z_min=0.001, z_max=10)

def generate_samples(size, dict_name):

    # draw masses from FIGARO population
    samples_3d = rvs_median(population_densities_3d, size=size)

    log_mass_1 = samples_3d[:, 0]
    log_mass_2 = samples_3d[:, 1]

    # Physical masses -- used ONLY for chirp-mass calculation
    mass_1 = 10**(log_mass_1)
    mass_2 = 10**(log_mass_2)

    # draw redshift from merger-rate density
    zs_samples = cbc.merger_rate_density_based_source_redshift.rvs(size=size)

    # luminosity distance
    luminosity_distance = ler.luminosity_distance.function(zs_samples)

    # other parameters
    T = 365 * 24 * 3600
    t_start = 0

    geocent_time = np.random.uniform(t_start, t_start + T, size)
    ra = np.random.uniform(0, 2*np.pi, size)
    phase = np.random.uniform(0, 2*np.pi, size)
    psi = np.random.uniform(0, np.pi, size)

    phi_12 = np.random.uniform(0, 2*np.pi, size)
    phi_jl = np.random.uniform(0, 2*np.pi, size)

    a_1 = np.random.uniform(0, 0.99, size)
    a_2 = np.random.uniform(0, 0.99, size)

    # isotropic distributions
    theta_jn = np.arccos(np.random.uniform(-1, 1, size))
    tilt_1 = np.arccos(np.random.uniform(-1, 1, size))
    tilt_2 = np.arccos(np.random.uniform(-1, 1, size))
    dec = np.arcsin(np.random.uniform(-1, 1, size))

    # construct dictionary
    dict_name = {
        "zs": zs_samples,
        "geocent_time": geocent_time,
        "ra": ra,
        "dec": dec,
        "phase": phase,
        "luminosity_distance": luminosity_distance,
        "psi": psi,
        "theta_jn": theta_jn,
        "a_1": a_1,
        "a_2": a_2,
        "mass_1": samples_3d[:, 0],
        "mass_2": samples_3d[:, 1],
        "tilt_1": tilt_1,
        "tilt_2": tilt_2,
        "phi_12": phi_12,
        "phi_jl": phi_jl,
    }

    # finite values
    mask = np.ones(size, dtype=bool)

    for value in dict_name.values():
        mask &= np.isfinite(value)

    # physical constraints
    mask &= dict_name["zs"] >= 0
    mask &= dict_name["luminosity_distance"] >= 0
    mask &= dict_name["mass_1"] >= dict_name["mass_2"]

    # filter
    dict_name = {
        key: value[mask]
        for key, value in dict_name.items()
    }

    return dict_name

print("Drawing samples from the population distribution...", flush=True)

unlensed_params = generate_samples(
    size=10000000,
    dict_name="unlensed_params"
)

w_grid = np.load("/Users/natalieyyyyy/Projects/w_grid.npy", allow_pickle=True)
chirp_mass_grid = np.load("/Users/natalieyyyyy/Projects/chirp_mass_grid.npy", allow_pickle=True)

w_func = interp1d(
    chirp_mass_grid,
    w_grid,
    kind="linear",
    fill_value="extrapolate",
    bounds_error=False
)

# global upper bound for the weight
chirp_mass_grid_dense = np.linspace(chirp_mass_grid.min(), chirp_mass_grid.max(), 10000)
w_max = 1.05 * np.nanmax(w_func(chirp_mass_grid_dense))

print("w_max =", w_max)

batch_n = 1000
target_n = 1000
accepted = []
n_acc_total = 0

while n_acc_total < target_n:

    cand = generate_samples(
        size=batch_n,
        dict_name="cand"
    )

    if len(cand["zs"]) == 0:
        continue

    # Convert log masses to physical masses
    mass_1 = 10**(cand["mass_1"])
    mass_2 = 10**(cand["mass_2"])

    # Chirp mass must be calculated in physical mass space
    mc = (
        (mass_1 * mass_2) ** (3/5)
        / (mass_1 + mass_2) ** (1/5)
    )

    # only use masses within the range of the weight function
    valid_mc = (
        np.isfinite(mc)
        & (mc >= chirp_mass_grid.min())
        & (mc <= chirp_mass_grid.max())
    )

    if not np.any(valid_mc):
        continue

    # normalized acceptance probability
    p_acc = np.asarray(
        w_func(mc[valid_mc]) / w_max,
        dtype=float
    )

    if np.any(p_acc < 0) or np.any(p_acc > 1):
        raise RuntimeError("Invalid acceptance probability.")

    # draw uniformly between 0 and the batch maximum
    u = np.random.uniform(0, 1, len(p_acc))

    # accept valid events
    keep_valid = u < p_acc

    # map back to the full candidate array
    keep = np.zeros(len(mc), dtype=bool)
    keep[valid_mc] = keep_valid

    accepted_batch = {
        key: value[keep]
        for key, value in cand.items()
    }

    n_acc = len(accepted_batch["zs"])

    if n_acc > 0:
        accepted.append(accepted_batch)
        n_acc_total += n_acc

    print(
        f"Accepted: {n_acc}; "
        f"total: {n_acc_total}/{target_n}"
    )

# combine accepted batches
new_params = {}

for key in accepted[0]:
    new_params[key] = np.concatenate(
        [batch[key] for batch in accepted]
    )

# exactly target_n events
for key in new_params:
    new_params[key] = new_params[key][:target_n]

print("final catalogue size:", len(new_params["zs"]))

samples_3d = np.column_stack([new_params["mass_1"], new_params["mass_2"], new_params["zs"]]).astype(float)
samples_3d = samples_3d[np.all(np.isfinite(samples_3d), axis=1)]

lower = np.min(samples_3d, axis=0) - 0.5
upper = np.max(samples_3d, axis=0) + 0.5
bounds = list(zip(lower, upper))

prior_pars = get_priors(bounds=bounds, samples=samples_3d, probit=False)
mix_3d = DPGMM(bounds=bounds, prior_pars=prior_pars, probit=False)

n_draws = 10 # for testing, can be increased to 100 or even higher for final version
draws = []
for _ in tqdm(range(n_draws)):
    draws.append(mix_3d.density_from_samples(samples_3d))

fig = plot_multidim(draws, labels=[r"$\log m_1$", r"$\log m_2$", r"$z_s$"], save = False)

axs = np.array(fig.axes).reshape(3, 3)

# fig.suptitle(r"Reweighted Component Masses Population", fontsize=18, y=1.00)
fig.subplots_adjust(top=0.96, wspace=0.07, hspace=0.07)

# Set font sizes for axes labels and ticks
for ax in fig.axes:
    ax.xaxis.label.set_size(14)
    ax.yaxis.label.set_size(14)
    ax.tick_params(axis='both', labelsize=12)

# Adjust the limits of the diagonal and off-diagonal plots based on the data
for i in range(2):

    # Set number of major ticks on the parameter axis
    axs[i, i].xaxis.set_major_locator(MaxNLocator(nbins=5))
    axs[i, i].xaxis.set_minor_locator(AutoMinorLocator(2))

    for j in range(i + 1, 3):

        # x-axis is the same parameter, so use the same ticks
        axs[j, i].xaxis.set_major_locator(MaxNLocator(nbins=5))
        axs[j, i].xaxis.set_minor_locator(AutoMinorLocator(2))

    for j in range(i + 1, 3):

        # x-axis is the same parameter, so use the same ticks
        axs[j, i].xaxis.set_major_locator(MaxNLocator(nbins=5))
        axs[j, i].xaxis.set_minor_locator(AutoMinorLocator(2))
    for j in range(i):

        # y-axis is the same parameter, so use the same ticks
        axs[i, j].yaxis.set_major_locator(MaxNLocator(nbins=5))
        axs[i, j].yaxis.set_minor_locator(AutoMinorLocator(2))

# Set font sizes for legend texts
for ax in fig.axes:
    if ax.legend_ is not None:
        for text in ax.legend_.get_texts():
            text.set_fontsize(14)

fig.savefig(f"/Users/natalieyyyyy/Projects/results/population_reweighted_m_1_m_2_log.pdf", dpi=300, bbox_inches="tight")


In [ ]:
# not needed as switched to 3d

# population density (2d; reweighted)

import numpy as np
from scipy.interpolate import interp1d
from pathlib import Path
from figaro.mixture import DPGMM
from figaro.utils import get_priors
from figaro.plot import plot_multidim
from tqdm import tqdm

from figaro.utils import rvs_median
from ler import LeR
from ler.gw_source_population import CBCSourceRedshiftDistribution
ler = LeR(spin_precession=True)

from matplotlib.ticker import MaxNLocator, AutoMinorLocator

np.random.seed(42)

population_densities_2d = np.load("/Users/natalieyyyyy/Projects/intermediates/population_densities_2d.npy", allow_pickle=True)
cbc = CBCSourceRedshiftDistribution(z_min=0.001, z_max=10)

def generate_samples(size, dict_name):

    # draw masses from FIGARO population
    samples_2d = rvs_median(population_densities_2d, size=size)

    # draw redshift from merger-rate density
    zs_samples = cbc.merger_rate_density_based_source_redshift.rvs(size=size)

    # luminosity distance
    luminosity_distance = ler.luminosity_distance.function(zs_samples)

    # other parameters
    T = 365 * 24 * 3600
    t_start = 0

    geocent_time = np.random.uniform(t_start, t_start + T, size)
    ra = np.random.uniform(0, 2*np.pi, size)
    phase = np.random.uniform(0, 2*np.pi, size)
    psi = np.random.uniform(0, np.pi, size)

    phi_12 = np.random.uniform(0, 2*np.pi, size)
    phi_jl = np.random.uniform(0, 2*np.pi, size)

    a_1 = np.random.uniform(0, 0.99, size)
    a_2 = np.random.uniform(0, 0.99, size)

    # isotropic distributions
    theta_jn = np.arccos(np.random.uniform(-1, 1, size))
    tilt_1 = np.arccos(np.random.uniform(-1, 1, size))
    tilt_2 = np.arccos(np.random.uniform(-1, 1, size))
    dec = np.arcsin(np.random.uniform(-1, 1, size))

    # construct dictionary
    dict_name = {
        "zs": zs_samples,
        "geocent_time": geocent_time,
        "ra": ra,
        "dec": dec,
        "phase": phase,
        "luminosity_distance": luminosity_distance,
        "psi": psi,
        "theta_jn": theta_jn,
        "a_1": a_1,
        "a_2": a_2,
        "mass_1": samples_2d[:, 0],
        "mass_2": samples_2d[:, 1],
        "tilt_1": tilt_1,
        "tilt_2": tilt_2,
        "phi_12": phi_12,
        "phi_jl": phi_jl,
    }

    # finite values
    mask = np.ones(size, dtype=bool)

    for value in dict_name.values():
        mask &= np.isfinite(value)

    # physical constraints
    mask &= dict_name["zs"] >= 0
    mask &= dict_name["mass_1"] >= 0
    mask &= dict_name["mass_2"] >= 0
    mask &= dict_name["luminosity_distance"] >= 0
    mask &= dict_name["mass_1"] >= dict_name["mass_2"]

    # filter
    dict_name = {
        key: value[mask]
        for key, value in dict_name.items()
    }

    return dict_name

print("Drawing samples from the population distribution...", flush=True)

unlensed_params = generate_samples(
    size=10000000,
    dict_name="unlensed_params"
)

w_grid = np.load("/Users/natalieyyyyy/Projects/w_grid.npy", allow_pickle=True)
chirp_mass_grid = np.load("/Users/natalieyyyyy/Projects/chirp_mass_grid.npy", allow_pickle=True)

w_func = interp1d(
    chirp_mass_grid,
    w_grid,
    kind="linear",
    fill_value="extrapolate",
    bounds_error=False
)

# global upper bound for the weight
chirp_mass_grid_dense = np.linspace(chirp_mass_grid.min(), chirp_mass_grid.max(), 10000)
w_max = 1.05 * np.nanmax(w_func(chirp_mass_grid_dense))

print("w_max =", w_max)

batch_n = 1000
target_n = 1000
accepted = []
n_acc_total = 0

while n_acc_total < target_n:

    cand = generate_samples(
        size=batch_n,
        dict_name="cand"
    )

    if len(cand["zs"]) == 0:
        continue

    # chirp mass
    mc = (
        (cand["mass_1"] * cand["mass_2"]) ** (3/5)
        / (cand["mass_1"] + cand["mass_2"]) ** (1/5)
    )

    # only use masses within the range of the weight function
    valid_mc = (
        np.isfinite(mc)
        & (mc >= chirp_mass_grid.min())
        & (mc <= chirp_mass_grid.max())
    )

    if not np.any(valid_mc):
        continue

    # normalized acceptance probability
    p_acc = np.asarray(
        w_func(mc[valid_mc]) / w_max,
        dtype=float
    )

    if np.any(p_acc < 0) or np.any(p_acc > 1):
        raise RuntimeError("Invalid acceptance probability.")

    # draw uniformly between 0 and the batch maximum
    u = np.random.uniform(0, 1, len(p_acc))

    # accept valid events
    keep_valid = u < p_acc

    # map back to the full candidate array
    keep = np.zeros(len(mc), dtype=bool)
    keep[valid_mc] = keep_valid

    accepted_batch = {
        key: value[keep]
        for key, value in cand.items()
    }

    n_acc = len(accepted_batch["zs"])

    if n_acc > 0:
        accepted.append(accepted_batch)
        n_acc_total += n_acc

    print(
        f"Accepted: {n_acc}; "
        f"total: {n_acc_total}/{target_n}"
    )

# combine accepted batches
new_params = {}

for key in accepted[0]:
    new_params[key] = np.concatenate(
        [batch[key] for batch in accepted]
    )

# exactly target_n events
for key in new_params:
    new_params[key] = new_params[key][:target_n]

print("final catalogue size:", len(new_params["zs"]))

samples_2d = np.column_stack([new_params["mass_1"], new_params["mass_2"]]).astype(float)
samples_2d = samples_2d[np.all(np.isfinite(samples_2d), axis=1)]

lower = np.min(samples_2d, axis=0) - 0.5
upper = np.max(samples_2d, axis=0) + 0.5
bounds = list(zip(lower, upper))

prior_pars = get_priors(bounds=bounds, samples=samples_2d, probit=False)
mix_2d = DPGMM(bounds=bounds, prior_pars=prior_pars, probit=False)

n_draws = 10 # for testing, can be increased to 100 or even higher for final version
draws = []
for _ in tqdm(range(n_draws)):
    draws.append(mix_2d.density_from_samples(samples_2d))

fig = plot_multidim(draws, labels=[r"$m_1$", r"$m_2$"], save = False)

axs = np.array(fig.axes).reshape(2, 2)

# fig.suptitle(r"Reweighted Component Masses Population", fontsize=18, y=1.00)
fig.subplots_adjust(top=0.96, wspace=0.07, hspace=0.07)

# Set font sizes for axes labels and ticks
for ax in fig.axes:
    ax.xaxis.label.set_size(14)
    ax.yaxis.label.set_size(14)
    ax.tick_params(axis='both', labelsize=12)

# Adjust the limits of the diagonal and off-diagonal plots based on the data
for i in range(2):

    # Set number of major ticks on the parameter axis
    axs[i, i].xaxis.set_major_locator(MaxNLocator(nbins=5))
    axs[i, i].xaxis.set_minor_locator(AutoMinorLocator(2))

    for j in range(i + 1, 2):

        # x-axis is the same parameter, so use the same ticks
        axs[j, i].xaxis.set_major_locator(MaxNLocator(nbins=5))
        axs[j, i].xaxis.set_minor_locator(AutoMinorLocator(2))

    for j in range(i + 1, 2):

        # x-axis is the same parameter, so use the same ticks
        axs[j, i].xaxis.set_major_locator(MaxNLocator(nbins=5))
        axs[j, i].xaxis.set_minor_locator(AutoMinorLocator(2))
    for j in range(i):

        # y-axis is the same parameter, so use the same ticks
        axs[i, j].yaxis.set_major_locator(MaxNLocator(nbins=5))
        axs[i, j].yaxis.set_minor_locator(AutoMinorLocator(2))

# Set font sizes for legend texts
for ax in fig.axes:
    if ax.legend_ is not None:
        for text in ax.legend_.get_texts():
            text.set_fontsize(14)

fig.savefig(f"/Users/natalieyyyyy/Projects/results/population_reweighted_m_1_m_2.pdf", dpi=300, bbox_inches="tight")


In [ ]:
# lensed population density (n-image system) (fig 5, 6, 7 in new paper)

# parameters:
# log10 relative effective luminosity distance
# log10 time delay in seconds

import json
from figaro.mixture import DPGMM
from figaro.utils import get_priors
from tqdm import tqdm
from matplotlib.ticker import MaxNLocator, AutoMinorLocator
from scipy.stats import gaussian_kde


# ============================================================
# SETTINGS
# ============================================================

n_images = 4  # 2, 3, or 4
n_draws = 10  # for testing, can be increased for final version

if n_images not in [2, 3, 4]:
    raise ValueError("n_images must be 2, 3, or 4.")

ndim = 2 * (n_images - 1)

dir = Path("/Users/natalieyyyyy/Projects")


# ============================================================
# LOAD EVENTS
# ============================================================

print("Loading events...", flush=True)

with open(dir / "lensed_param_detectable.json", "r") as f:
    data = json.load(f)

n_events = len(data["n_images"])
print(f"Loaded {n_events} events", flush=True)


# ============================================================
# CONSTRUCT SAMPLES
# ============================================================

samples_nd = []

for i in range(n_events):

    if int(data["n_images"][i]) != n_images:
        continue

    magnifications = np.asarray(data["magnifications"][i], dtype=float)
    time_delays = np.asarray(data["time_delays"][i], dtype=float)

    # Keep only the actual images; remaining entries are zero padding
    magnifications = magnifications[:n_images]
    time_delays = time_delays[:n_images]

    if len(magnifications) != n_images or len(time_delays) != n_images:
        continue

    if not (np.all(np.isfinite(magnifications)) and np.all(np.isfinite(time_delays))):
        continue

    # Sort images by arrival time
    order = np.argsort(time_delays)
    magnifications = magnifications[order]
    time_delays = time_delays[order]

    # Magnifications must be nonzero
    if np.any(np.abs(magnifications) <= 0):
        continue

    # Relative effective luminosity distances
    # log10(dL_eff,i / dL_eff,1) = 0.5 * log10(|mu_1| / |mu_i|)
    mu_1 = np.abs(magnifications[0])
    log_relative_dL = 0.5 * np.log10(mu_1 / np.abs(magnifications[1:]))

    # Relative arrival times in seconds
    delta_t = time_delays[1:] - time_delays[0]

    if np.any(delta_t <= 0):
        continue

    log_time_delays = np.log10(delta_t)

    # First relative distances, then time delays
    samples_nd.append(np.concatenate([log_relative_dL, log_time_delays]))


samples_nd = np.asarray(samples_nd, dtype=float).reshape(-1, ndim)
samples_nd = samples_nd[np.all(np.isfinite(samples_nd), axis=1)]

print(f"{n_images}-image systems: {len(samples_nd)} valid events", flush=True)
print("samples shape:", samples_nd.shape, flush=True)


# ============================================================
# DPGMM
# ============================================================

print("Building DPGMM...", flush=True)

if len(samples_nd) == 0:
    raise ValueError(f"No valid {n_images}-image samples found.")

lower = np.min(samples_nd, axis=0) - 0.5
upper = np.max(samples_nd, axis=0) + 0.5
bounds = list(zip(lower, upper))

print("bounds:", bounds, flush=True)

prior_pars = get_priors(bounds=bounds, samples=samples_nd, probit=True)
mix = DPGMM(bounds=bounds, prior_pars=prior_pars, probit=True)

draws = []

for _ in tqdm(range(n_draws), desc="DPGMM"):
    draws.append(mix.density_from_samples(samples_nd))


# ============================================================
# SAVE POPULATION DENSITIES
# ============================================================

(dir / "intermediates").mkdir(parents=True, exist_ok=True)

np.save(
    dir / "intermediates" / f"population_densities_l{n_images}_dL_dt.npy",
    draws,
    allow_pickle=True
)


# ============================================================
# PLOT
# ============================================================

print("Loading figure...", flush=True)

labels = [
    rf"$\log_{{10}}(d_{{L,{i}}}^{{\mathrm{{eff}}}}/d_{{L,1}}^{{\mathrm{{eff}}}})$"
    for i in range(2, n_images + 1)
]

labels += [
    rf"$\log_{{10}}(\Delta t_{{{i}}}/\mathrm{{s}})$"
    for i in range(2, n_images + 1)
]

fig = plot_multidim(
    draws,
    samples=samples_nd,
    labels=labels,
    show=False,
    median_label=r"$\mathrm{DPGMM}$"
)

axs = np.array(fig.axes).reshape(ndim, ndim)


# ============================================================
# FIGURE FORMATTING
# ============================================================

fig.subplots_adjust(top=0.96, wspace=0.07, hspace=0.07)

# Slightly smaller fonts for higher-dimensional plots
if ndim == 2:
    label_size = 16
    tick_size = 13
    legend_size = 14
elif ndim == 4:
    label_size = 14
    tick_size = 11
    legend_size = 12
else:
    label_size = 12
    tick_size = 9
    legend_size = 10

for ax in fig.axes:
    ax.xaxis.label.set_size(label_size)
    ax.yaxis.label.set_size(label_size)
    ax.tick_params(axis="both", labelsize=tick_size)


# ============================================================
# AXIS TICKS
# ============================================================

for i in range(ndim):
    axs[i, i].xaxis.set_major_locator(MaxNLocator(nbins=5))
    axs[i, i].xaxis.set_minor_locator(AutoMinorLocator(2))

    for j in range(i + 1, ndim):
        axs[j, i].xaxis.set_major_locator(MaxNLocator(nbins=5))
        axs[j, i].xaxis.set_minor_locator(AutoMinorLocator(2))

    for j in range(i):
        axs[i, j].yaxis.set_major_locator(MaxNLocator(nbins=5))
        axs[i, j].yaxis.set_minor_locator(AutoMinorLocator(2))


# ============================================================
# LEGEND
# ============================================================

for ax in fig.axes:
    if ax.legend_ is not None:
        for text in ax.legend_.get_texts():
            text.set_fontsize(legend_size)


# ============================================================
# SAVE
# ============================================================

output_path = dir / "results" / f"lensed_population_l{n_images}_dL_dt.pdf"
output_path.parent.mkdir(parents=True, exist_ok=True)

fig.savefig(output_path, dpi=300, bbox_inches="tight")

print(f"Done! Figure saved to {output_path}", flush=True)

Bayes Factor / Posterior Odds Plots

In [ ]:
# log bayes factor and log posterior odds histograms (fig 8 in new paper)

# ============================================================
# SETTINGS
# ============================================================

from matplotlib.ticker import MultipleLocator

dir = Path("/Users/natalieyyyyy/Projects")

n_images_list = [2, 3, 4]

prior_odds = {
    2: 1.9e-06,
    3: 1.5e-09,
    4: 3.9e-11
}

colors = {
    2: "tab:blue",
    3: "tab:orange",
    4: "tab:green"
}

labels = {
    2: r"$2\ \mathrm{images}$",
    3: r"$3\ \mathrm{images}$",
    4: r"$4\ \mathrm{images}$"
}


# ============================================================
# LOAD RESULTS
# ============================================================

all_results = {}

for n_images in n_images_list:

    results = pd.read_csv(
        dir / "results" /
        f"bayes_factor_results_{n_images}.csv"
    )

    valid_results = results[
        np.isfinite(results["log10_BLU"])
        & (results["QuickCheck"] == 0)
    ].copy()

    # log10 posterior odds
    valid_results["log10_posterior_odds"] = (
        valid_results["log10_BLU"]
        + np.log10(prior_odds[n_images])
    )

    all_results[n_images] = valid_results

    print(
        f"{n_images}-image systems: "
        f"{len(valid_results)} valid results",
        flush=True
    )

# ============================================================
# COMMON AXIS RANGES AND BINS
# ============================================================

xmin, xmax = -40, 30
bins = np.linspace(xmin, xmax, 51)

max_count = 0
for n_images in n_images_list:
    for column in ["log10_BLU", "log10_posterior_odds"]:
        counts, _ = np.histogram(all_results[n_images][column], bins=bins)
        max_count = max(max_count, counts.max())

ymax = 10 * np.ceil(max_count * 1.1 / 10)

# ============================================================
# PLOT: BAYES FACTORS AND POSTERIOR ODDS
# ============================================================

fig, axs = plt.subplots(
    2, 1,
    figsize=(8, 10),
    sharex=True,
    sharey=True
)

columns = ["log10_BLU", "log10_posterior_odds"]

xlabels = [
    r"$\log_{10} B_U^L$",
    r"$\log_{10} O_U^L$"
]

for ax, column, xlabel in zip(axs, columns, xlabels):

    for n_images in n_images_list:
        data = all_results[n_images]

        ax.hist(
            data[column],
            bins=bins,
            color=colors[n_images],
            alpha=0.5,
            label=labels[n_images]
        )

        median = np.median(data[column])

        ax.axvline(
            median,
            color=colors[n_images],
            linestyle="--",
            linewidth=1,
            alpha=0.8
        )

    ax.set_xlim(xmin, xmax)
    ax.set_ylim(0, ymax)
    ax.set_ylabel("Number of systems", fontsize=16)
    ax.tick_params(axis="both", labelsize=13)
    ax.set_xticks(np.arange(-40, 31, 10))
    ax.legend(fontsize=13, frameon=False)

# Keep both x-axis labels visible
axs[0].set_xlabel(xlabels[0], fontsize=16)
axs[1].set_xlabel(xlabels[1], fontsize=16)

# sharex=True hides upper x tick labels by default
axs[0].tick_params(axis="x", labelbottom=True)

fig.tight_layout()

fig.savefig(
    dir / "results" / "bayes_factor_and_posterior_odds_comparison.pdf",
    dpi=300,
    bbox_inches="tight"
)



In [ ]:
# scatter plot of log10 Bayes factor vs component masses (fig 9 in new paper)

# ============================================================
# SETTINGS
# ============================================================

from matplotlib.patches import Patch

dir = Path("/Users/natalieyyyyy/Projects")

n_images_list = [2, 3, 4]

colors = {
    2: "tab:blue",
    3: "tab:orange",
    4: "tab:green"
}

labels = {
    2: r"$2\ \mathrm{images}$",
    3: r"$3\ \mathrm{images}$",
    4: r"$4\ \mathrm{images}$"
}


# ============================================================
# LOAD DATA
# ============================================================

all_data = {}

for n_images in n_images_list:

    file_path = (
        dir /
        "intermediates" /
        f"bayes_factor_masses_{n_images}.csv"
    )

    data = pd.read_csv(file_path)

    # Keep finite values only
    data = data[
        np.isfinite(data["log10_BLU"])
        & np.isfinite(data["mass_1"])
        & np.isfinite(data["mass_2"])
    ].copy()

    all_data[n_images] = data

    print(
        f"Loaded {len(data)} {n_images}-image systems",
        flush=True
    )


# ============================================================
# PLOT
# ============================================================

fig, axs = plt.subplots(
    1,
    2,
    figsize=(10, 4.5)
)


for n_images in n_images_list:

    data = all_data[n_images]


    # --------------------------------------------------------
    # mass 1
    # --------------------------------------------------------

    axs[0].scatter(
        data["mass_1"],
        data["log10_BLU"],
        s=8,
        alpha=0.4,
        color=colors[n_images]
    )


    # --------------------------------------------------------
    # mass 2
    # --------------------------------------------------------

    axs[1].scatter(
        data["mass_2"],
        data["log10_BLU"],
        s=8,
        alpha=0.4,
        color=colors[n_images]
    )


# ============================================================
# AXIS LABELS
# ============================================================

axs[0].set_xlabel(
    r"$m_1\,[M_\odot]$",
    fontsize=16
)

axs[0].set_ylabel(
    r"$\log_{10}\mathcal{B}$",
    fontsize=16
)


axs[1].set_xlabel(
    r"$m_2\,[M_\odot]$",
    fontsize=16
)

axs[1].set_ylabel(
    r"$\log_{10}\mathcal{B}$",
    fontsize=16
)


# ============================================================
# FORMATTING
# ============================================================

for ax in axs:

    ax.tick_params(
        axis="both",
        labelsize=13
    )


# ============================================================
# LEGEND
# ============================================================

legend_handles = [
    Patch(
        facecolor=colors[n_images],
        label=labels[n_images]
    )
    for n_images in n_images_list
]

axs[0].legend(
    handles=legend_handles,
    fontsize=13,
    frameon=False,
    loc="upper right"
)


# ============================================================
# SAVE
# ============================================================

fig.tight_layout()

fig.savefig(
    dir /
    "results" /
    "log_bayes_factor_vs_component_masses_all_images.pdf",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

In [ ]:
# not needed unless want to plot the histogram of log10 Bayes factor and posterior odds for each number of images separately

# log bayes factor

results = pd.read_csv("/Users/natalieyyyyy/Projects/results/bayes_factor_results_4.csv")

valid_results = results[
    results["log10_BLU"].notna()
    & (results["QuickCheck"] == 0)
]

posterior_odds = valid_results["log10_BLU"] + np.log10()
# 2-image prior odds: 1.9e-06
# 3-image prior odds: 1.5e-09
# 4-image prior odds: 3.9e-11

fig, ax = plt.subplots(figsize=(8, 5))

ax.hist(valid_results["log10_BLU"], bins=50, color='skyblue', alpha=0.7)
ax.hist(posterior_odds, bins=50, color='lightcoral', alpha=0.7)

ax.set_xlabel(r"$\log_{10} B_U^L$")
ax.set_ylabel("Number of systems")

plt.tight_layout()
plt.savefig("/Users/natalieyyyyy/Projects/results/log10_BLU_histogram.pdf", dpi=300, bbox_inches="tight")